In [1]:
#######################################################
# Armazena os pareceres de indeferimento localmente de todos os pedidos que estão na carga e que tem indeferimento

import os, re
import json
import requests

In [2]:
comentarios = """
rode xampp local
limpe tabela carga e depois faça o import carga.csv
Anaconda ambiente python 

1. CRIA NOVOS REGISTROS EM ANTERIORIDADES_DESC E ATUALIZA CAMPO DESCRICAO COM DISCUSSÃO SOBRE ATIVIDADE INVENTIVA. Usa LLM 
(insert_anterioridades_desc.ipynb)
comando = f"select * from carga where numero<>'NUMERO' and numero not in (select numero from anterioridades_desc) and numero in (select numero from arquivados where despacho='12.2')"
# ******************************************INSERT ANTERIORIDADES_DESC CAMPO DESCRICAO COM DISCUSSÃO DA ATIVIDADE INVENTIVA
normalmente são uns 30 registros novos que aparecem na carga a cada semana
convem limpar o arquivo descricao.sql que é onde são gravadas as saídas deste comando

2. SALVA PARECERES DE INDEFERIMENTO E SALVA PETIÇÕES 214 
(salvar_pareceres.ipynb)
# Armazena os pareceres de indeferimento localmente de todos os pedidos que estão na carga e que tem indeferimento

3. ATUALIZA CAMPO CONCLUSAO com o texto literal da conclusao do parecer de indeferimento. Não usa LLM
(insert_anterioridades_desc.ipynb)
comando = f"select * from anterioridades_desc where conclusao='';"
## update anterioridades_desc set conclusao='{conclusao}'
procure por: SELECT * FROM `anterioridades_desc` WHERE conclusao like '%Código:%'
procure por: SELECT * FROM `anterioridades_desc` WHERE conclusao like '%Rio de Janeiro:%'
e apague o lixo, bem como caracteres especiais

4. CRIA NOVOS REGISTROS EM ANTERIORIDADES 
(insert_anterioridades_desc.ipynb)
comando = f"select * from carga where numero not in (select numero from anterioridades) and numero in (select numero from arquivados where despacho='12.2')"
sql = f"INSERT IGNORE INTO anterioridades

5. ATUALIZA CAMPO RAZOES COM A CONCLUSAO DE INDEFERIMENTO CITANDO OS DOCUMENTOS DE ATIVIDADE INVENTIVA
(insert_anterioridades_desc.ipynb)
comando = f"select * from anterioridades_desc where razoes='' AND numero in (select numero from carga) and numero in (select numero from arquivados where despacho='12.2')"
esta rotina usa LLM e estava resultando muitos erros
melhor usar para calcular o campo razoes:
https://cientistaspatentes.com.br/central/control.php?action=190&op=13
e depois rodar op=12 para montar historico_pedido (usando razoes)
https://cientistaspatentes.com.br/central/control.php?action=190&op=12

5. ATUALIZA CAMPO INCOERENCIA. Usa LLM
(insert_anterioridades_desc.ipynb)
comando = f"select * from anterioridades_desc where incoerencia='' and numero in (select numero from carga);"
## UPDATE anterioridades_desc campo incoerencia

7. ATUALIZA CAMPO RESUMO_RECURSO. Usa LLM
RESUMIR_PETICAO_214.IPYNB
select * from anterioridades_desc where resumo_recurso='' and numero in (select numero from carga)
UPDATE anterioridades_desc set resumo_recurso='{resumo}

8. ATUALIZA CAMPO compara_docs. Usa LLM
comparacao_docs.IPYNB
tem antes que baixar as petições PDF desta rotina salvar_pareceres.ipynb
comando = f"select * from anterioridades_desc where comparacao_docs='' and numero in (select numero from carga) limit 2;"
UPDATE anterioridades_desc set comparacao_docs='{output}' WHERE numero='{numero}
"""


In [2]:
import json
import requests

def conectar_siscap(url,return_json=False):
    headers = {
        "Accept": "application/json",
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }
    response = requests.get(url,headers=headers,verify=False)
    if response.status_code == 200:
        if return_json:
            data = response.json()
            json_data = json.dumps(data, indent=4)
            return(json_data)
        else:
            return response.text
    else:
        return(f"Erro: {response.status_code}")

In [3]:
# aplique os INSERTs obtidos em insert_anterioridades_desc na tabela anterioridades_desc local do computador e no hostgator

import mysql.connector
conexao = mysql.connector.connect(host='localhost',user='root',password='',database='producao')
cursor = conexao.cursor()

In [4]:
TIPO_CHOICES2 = {
    "exigencia": "exigência (6.1)",
    "ciencia de parecer": "ciência de parecer (7.1)",
    "indeferimento": "indeferimento (9.2)",
    "200": "depósito (200)",
    "202": "publicação antecipada (202)",
    "203": "exame (203)",
    "207": "cumprimento exigência (207)",
    "210": "subsídios ao exame (210)",
    "260": "outras petições (260)",
    "272": "manifestação ao parecer técnico (272)",
    "280": "cumprimento de exigência (280)",
    "281": "manifestação (281)",
    "9.2": "indeferimento (9.2)",
    "214": "recurso (214)",
    "121": "recurso exigência (121)",
    "280": "cumprimento exigência (280)",
    "111": "recurso negado (111)",
    "100.1": "recurso provido-reforma (100.1)",
    "100.2": "recurso provido-devolução (100.2)",
    "130": "recurso prejudicado (130)",
}

TIPO_CHOICES = [
        ('6.1', 'exigência (6.1)'),
        ('exigencia', 'exigência (6.1)'),
        ('7.1', 'ciência de parecer (7.1)'),
        ('ciencia de parecer', 'ciência de parecer (7.1)'),
        ('9.2', 'indeferimento (9.2)'),
        ('indeferimento', 'indeferimento (9.2)'),
        ('recurso provido-reforma 100.1', 'recurso provido-reforma (100.1)'),
        ('recurso provido-devolucao 100.2', 'recurso provido-devolução (100.2)'),
        ('recurso provido', 'recurso provido (100)'),
        ('recurso provido anvisa', 'recurso provido (100)'),
        ('recurso negado', 'recurso negado (111)'),
        ('recurso exigencia', 'recurso exigência (121)'),
        ('recurso ciencia', 'recurso ciência (121)'),
        ('recurso exigencia 121', 'recurso exigência (121)'),
        ('130', 'recurso prejudicado (130)'),
        ('200', 'depósito (200)'),
        ('202', 'publicação antecipada (202)'),
        ('203', 'pedido de exame de invenção (203)'),
        ('204', 'pedido de exame de modelo de utilidade (204)'),
        ('205', 'pedido de exame de certificado de adição (205)'),
        ('207', 'cumprimento exigência (207)'),
        ('210', 'subsídios ao exame (210)'),
        ('214', 'recurso administrativo (214)'),
        ('215', 'nulidade administrativa (215)'),
        ('216', 'contestação à nulidade (216)'),
        ('260', 'outras petições (260)'),
        ('272', 'manifestação sobre parecer técnico de recurso (272)'),
        ('280', 'cumprimento exigência de recurso (280)'),
        ('281', 'manifestação em primeira instância (281)'),
        ('282', 'manifestação sobre nulidade (282)'),
        ('284', 'pedido de exame de invenção via PCT com ISA/IPEA BR (284)'),
        ('285', 'pedido de exame de modelo de utilidade com ISA/IPEA BR (285)'),
        ('295', 'contrarrazões ao recurso (295)'),
        ('296', 'cumprimento exigência formal (296)'),
]

decisao = 'indeferimento'
resultado = [v for k, v in TIPO_CHOICES if k == decisao][0]
print(resultado)

def converter_data(data_iso):
    ano, mes, dia = data_iso.split("-")
    return f"{dia}/{mes}/{ano}"

indeferimento (9.2)


In [5]:
# leitura dos pareceres técnicos de exigencia, ciencia e indeferimento do SISCAP e salta TXT localmente
# faz a leitura dos pareceres no SISCAP. Não faz uso de LLM. certifique-se de ter a tabela anterioridades_desc atualizada

query = '"' + "mysql_query" + '"' ":" + '"' + f" * FROM carga" + '"'
#query = '"' + "mysql_query" + '"' ":" + '"' + f" numero FROM anterioridades_desc" + '"'
url = f"https://cientistaspatentes.com.br/apiphp/patents/query/?q={query}"
print(url)
json_data = conectar_siscap(url,return_json=True)
    
#json_data='{"patents": [{"numero":"PI0905487","prioridade":"BR","instancia":"2 exame","decisao":"indeferimento","prioritario":"0","cc1":"4","anulado":"0","codigo":"1340921","rpi":"2020-12-29","divisao":"dicel","etapa":"2"}]}'
json_data = re.sub(r'[\x00-\x08\x0B\x0C\x0E-\x1F]', '', json_data)
json_data = json_data.replace('\r', '')
#print(json_data)
data = json.loads(json_data)

for patent in data.get("patents", []):
    numero = patent.get("numero")
    if not numero or numero == "NUMERO":
        continue
        
    query_pedido = '"' + "mysql_query" + '"' ":" + '"' + f" * FROM pedido where decisao in ('indeferimento','9.2','exigencia','ciencia de parecer') and rpi is not null and numero='{numero}'" + '"'
    url = f"https://cientistaspatentes.com.br/apiphp/patents/query/?q={query_pedido}"
    print(url)

    codigo = None
    divisao = None
    try:
        json_data = conectar_siscap(url,return_json=True)
    except:
        pass

    if not json_data:
        continue

    if json_data:
        try:
            #json_data='{"patents": [{"numero":"PI0905487","prioridade":"BR","instancia":"2 exame","decisao":"indeferimento","prioritario":"0","cc1":"4","anulado":"0","codigo":"1340921","rpi":"2020-12-29","divisao":"dicel","etapa":"2"}]}'
            json_data = re.sub(r'[\x00-\x08\x0B\x0C\x0E-\x1F]', '', json_data)
            json_data = json_data.replace('\r', '')
            data_pedido = json.loads(json_data)
            #patents = data_pedido.get("patents", [])

            for i in range(0, len(data_pedido["patents"])):
                if i==2: break
                numero = data_pedido["patents"][i]["numero"]
                codigo = data_pedido["patents"][i]["codigo"]
                divisao = data_pedido["patents"][i]["divisao"]
                decisao = data_pedido["patents"][i]["decisao"]
                data_rpi = data_pedido["patents"][i]["rpi"]
                nova_data = converter_data(data_rpi) if data_rpi else ""

                if not codigo or not divisao or divisao == "sanot":
                    continue

                url = f"https://siscap.inpi.gov.br/adm/pareceres/{divisao}/{numero}{codigo}.txt"
                print(url)
            
                try:
                    texto_relatorio = conectar_siscap(url, return_json=False)
                except:
                    continue
                    
                if not texto_relatorio or len(texto_relatorio.strip()) < 50:
                    continue
                    
                caminho_do_arquivo = f"pareceres/{numero}{codigo}.txt"
                # os.makedirs(os.path.dirname(caminho_do_arquivo), exist_ok=True)

                resultado = [v for k, v in TIPO_CHOICES if k == decisao][0]
                output = numero + '\n' + "Parecer Técnico " + resultado + '\n'
                output = output + "Data de publicação na RPI: " + nova_data + '\n\n'
                texto_relatorio = output + texto_relatorio
                texto_relatorio = re.sub(r'(Pesquisador).*', r'\1', texto_relatorio, flags=re.S)

                if not os.path.exists(caminho_do_arquivo):
                    #os.makedirs(os.path.dirname(caminho_do_arquivo), exist_ok=True)
                    print(f"Arquivo novo criado: {caminho_do_arquivo}")
                    with open(caminho_do_arquivo, "w", encoding="utf-8") as arquivo:
                        arquivo.write(texto_relatorio)
                else:
                    print(f"Arquivo já existe, não sobrescrito: {caminho_do_arquivo}")   
                    
        except json.JSONDecodeError:
            pass  # JSON inválido → segue o fluxo sem abortar

https://cientistaspatentes.com.br/apiphp/patents/query/?q="mysql_query":" * FROM carga"


D:\Users\abrantes\InstallAnaconda\envs\python\lib\site-packages\urllib3\connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host 'cientistaspatentes.com.br'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


https://cientistaspatentes.com.br/apiphp/patents/query/?q="mysql_query":" * FROM pedido where decisao in ('indeferimento','9.2','exigencia','ciencia de parecer') and rpi is not null and numero='PI0921237'"


D:\Users\abrantes\InstallAnaconda\envs\python\lib\site-packages\urllib3\connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host 'cientistaspatentes.com.br'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


https://siscap.inpi.gov.br/adm/pareceres/dimol/PI09212371105868.txt


...

D:\Users\abrantes\InstallAnaconda\envs\python\lib\site-packages\urllib3\connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host 'cientistaspatentes.com.br'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


https://siscap.inpi.gov.br/adm/pareceres/difel/1120180099271592120.txt


D:\Users\abrantes\InstallAnaconda\envs\python\lib\site-packages\urllib3\connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host 'siscap.inpi.gov.br'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Arquivo já existe, não sobrescrito: pareceres/1120180099271592120.txt
https://siscap.inpi.gov.br/adm/pareceres/difel/1120180099271606649.txt
Arquivo já existe, não sobrescrito: pareceres/1120180099271606649.txt


D:\Users\abrantes\InstallAnaconda\envs\python\lib\site-packages\urllib3\connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host 'siscap.inpi.gov.br'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [5]:
import re
import hashlib
from collections import defaultdict

class DataAnonymizer:
    def __init__(self):
        # token -> valor original
        self.token_map = {}

        # valor original -> token (garante determinismo)
        self.reverse_map = {}

        # contadores por tipo
        self.token_counter = defaultdict(int)

    # ==============================
    # GERAÇÃO DE TOKEN
    # ==============================
    def _generate_token(self, tipo, valor):
        if not valor:
            return valor

        valor = valor.strip()

        # Determinístico: mesma string → mesmo token
        if valor in self.reverse_map:
            return self.reverse_map[valor]

        self.token_counter[tipo] += 1
        token = f"[{tipo}_{self.token_counter[tipo]}]"

        self.token_map[token] = valor
        self.reverse_map[valor] = token

        return token

    # ==============================
    # REMOÇÃO DE CABEÇALHOS
    # ==============================
    def remover_cabecalhos(self, texto):
        if not isinstance(texto, str):
            return ""
        padroes_remover = [
            r"Assinado digitalmente por.*",
            r"Documento assinado eletronicamente.*",
            r"Protocolo:\s*\d+",
            r"URL para download:.*",
            r"Hash de autenticação:.*",
        ]

        for padrao in padroes_remover:
            texto = re.sub(padrao, "", texto, flags=re.IGNORECASE)

        return texto

    # ==============================
    # ANONIMIZAÇÃO DE CPFs
    # ==============================
    def anonymize_cpfs(self, texto):
        if not isinstance(texto, str):
            return ""
        regex_cpf = r"\b\d{3}\.\d{3}\.\d{3}-\d{2}\b"

        def substituir(match):
            cpf = match.group()
            return self._generate_token("CPF", cpf)

        return re.sub(regex_cpf, substituir, texto)

    # ==============================
    # ANONIMIZAÇÃO DE CNPJ
    # ==============================
    def anonymize_cnpj(self, texto):
        if not isinstance(texto, str):
            return ""
        regex_cnpj = r"\b\d{2}\.\d{3}\.\d{3}/\d{4}-\d{2}\b"

        def substituir(match):
            cnpj = match.group()
            return self._generate_token("CNPJ", cnpj)

        return re.sub(regex_cnpj, substituir, texto)

    # ==============================
    # ANONIMIZAÇÃO DE PROCESSOS (9 dígitos)
    # ==============================
    def anonymize_processos(self, texto):
        if not isinstance(texto, str):
            return ""
        regex_processo = r"\b\d{9}\b"

        def substituir(match):
            processo = match.group()
            return self._generate_token("PROCESSO", processo)

        return re.sub(regex_processo, substituir, texto)

    # ==============================
    # ANONIMIZAÇÃO DE EMAIL
    # ==============================
    def anonymize_emails(self, texto):
        if not isinstance(texto, str):
            return ""
        regex_email = r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"

        def substituir(match):
            email = match.group()
            return self._generate_token("EMAIL", email)

        return re.sub(regex_email, substituir, texto)

    # ==============================
    # ANONIMIZAÇÃO DE NOMES SIMPLES (heurística)
    # ==============================
    def anonymize_nomes_maiusculos(self, texto):
        if not isinstance(texto, str):
            return ""
        # Heurística: nomes em caixa alta com pelo menos 2 palavras
        regex_nome = r"\b([A-ZÁÉÍÓÚÂÊÔÃÕÇ]{2,}(?:\s+[A-ZÁÉÍÓÚÂÊÔÃÕÇ]{2,})+)\b"

        def substituir(match):
            nome = match.group(1)
            return self._generate_token("PESSOA_NATURAL", nome)

        return re.sub(regex_nome, substituir, texto)


    # ==============================
    # DOCUMENTOS GENÉRICOS DE IDENTIFICAÇÃO
    # ==============================
    def anonymize_documentos_identificacao(self, texto):
        if not isinstance(texto, str):
            return ""
        regex = r"(CPF\/CNPJ|CPF|CNPJ)\s*:\s*([A-Z0-9\-\.\/]+)"
    
        def substituir(match):
            rotulo = match.group(1)
            valor = match.group(2)
            token = self._generate_token("DOC_ID", valor)
            return f"{rotulo}: {token}"
    
        return re.sub(regex, substituir, texto, flags=re.IGNORECASE)
        
    # ==============================
    # IDENTIFICADORES INTERNACIONAIS
    # ==============================
    def anonymize_identificadores_internacionais(self, texto):
        if not isinstance(texto, str):
            return ""
        regex = r"\b[A-Z]{2}\d{6,}\b"
    
        def substituir(match):
            valor = match.group()
            return self._generate_token("REGISTRO_INT", valor)
    
        return re.sub(regex, substituir, texto)

    # ==============================
    # ENDEREÇO
    # ==============================
    def anonymize_endereco(self, texto):
        if not isinstance(texto, str):
            return ""
        regex = r"(Endere[cç]o\s*:\s*)(.+)"
    
        def substituir(match):
            prefixo = match.group(1)   # "Endereço: "
            valor = match.group(2).strip()
    
            token = self._generate_token("ENDERECO", valor)
    
            return f"{prefixo}{token}"
    
        return re.sub(regex, substituir, texto, flags=re.IGNORECASE)

    # ==============================
    # REMOVER A LINHA INTEIRA QUE TENHA CEP
    # ==============================
    def anonymize_remover_linhas_com_cep(self, texto):
        if not isinstance(texto, str):
            return ""
        linhas = texto.splitlines()
    
        linhas_filtradas = [
            linha for linha in linhas
            if not re.search(r'\bCEP\b', linha, flags=re.IGNORECASE)
        ]
    
        return "\n".join(linhas_filtradas)
    
    # ==============================
    # REMOVER CABEÇALHO DAS PÁGINAS DA PETIÇÃO
    # ==============================
    def anonymize_remover_cabecalhos_pagina(self, texto):
        if not isinstance(texto, str):
            return ""
        linhas = texto.splitlines()
    
        linhas_filtradas = [
            linha for linha in linhas
            if not re.search(r'^\s*Peticao\b', linha, flags=re.IGNORECASE)
        ]
    
        return "\n".join(linhas_filtradas)

    # ==============================
    # TELEFONE
    # ==============================
    def anonymize_telefone(self, texto):
        if not isinstance(texto, str):
            return ""
        regex = r"(Telefone|Fax\s*:\s*)(.+)"
        regex = r"\b(Fone\/Fax|Fone|Telefone|Tel\.?|Fax)\b\s*:?\s*([\(\d][\d\.\-\)\s]+)"
    
        def substituir(match):
            prefixo = match.group(1)   # "Endereço: "
            valor = match.group(2).strip()
    
            token = self._generate_token("TELEFONE", valor)
    
            return f"{prefixo}{token}"
    
        return re.sub(regex, substituir, texto, flags=re.IGNORECASE)

    # ==============================
    # NOMES ROTULADOS
    # ==============================
    def anonymize_nomes_rotulados(self, texto):
        if not isinstance(texto, str):
            return ""
        regex = r"(Requerente|Técnico|Inventor|Procurador)\s*:\s*([A-ZÁÉÍÓÚÂÊÔÃÕÇ\s]+)"
    
        def substituir(match):
            rotulo = match.group(1)
            nome = match.group(2).strip()
            token = self._generate_token("PESSOA_NATURAL", nome)
            return f"{rotulo}: {token}"
    
        return re.sub(regex, substituir, texto)
        
    # ==============================
    # PIPELINE COMPLETO
    # ==============================
    def anonymize_texto(self, texto):
        texto = self.remover_cabecalhos(texto)
        texto = self.anonymize_cpfs(texto)
        texto = self.anonymize_cnpj(texto)
        texto = self.anonymize_emails(texto)
        texto = self.anonymize_processos(texto)
        #texto = self.anonymize_nomes_maiusculos(texto) # este critério estava retirando qualquer sequencia de maiusculos o que elimina título do pedido
        texto = self.anonymize_documentos_identificacao(texto)

        #texto = self.anonymize_identificadores_internacionais(texto)
        texto = self.anonymize_endereco(texto)
        texto = self.anonymize_telefone(texto)
        texto = self.anonymize_nomes_rotulados(texto)
        texto = self.anonymize_remover_linhas_com_cep(texto)
        texto = self.anonymize_remover_cabecalhos_pagina(texto)

        return texto

    
    # ==============================
    # DESTOKENIZAÇÃO
    # ==============================
    def deanonymize(self, texto):
        # Substitui tokens pelos valores originais
        for token, valor in self.token_map.items():
            texto = texto.replace(token, valor)
        return texto

    def iniciar_apos_recurso_207(self, texto):
        if not isinstance(texto, str):
            return ""
        padrao = r"(RESPOSTA\s+[AÀ]\s+EXIG[EÊ]NCIA\b|CUMPRIMENTO\s+DE\s+EXIG[EÊ]NCIA\b|RESPOSTA\s+[AÀ]\s+CI[EÊ]NCIA\b|MANIFESTA[ÇC][AÃ][O0]\s+(?:SOBRE\s+O|AO)\s+EXAME\b|ESCLARECIMENTOS?\b|E\s*S\s*C\s*L\s*A\s*R\s*E\s*C\s*I\s*M\s*E\s*N\s*T\s*O\s*S?|Excelent[ií]ssim[oa]\b|Ilustr[ií]ssim[oa]\s|Ilm[oa]\s+Senhor\s+|Ilm[oa]\s+Senhora\b)"
        match = re.search(padrao, texto, flags=re.IGNORECASE | re.DOTALL)
        if match:
            return texto[match.start():]
        else:
            return

    def iniciar_apos_recurso_210(self, texto):
        if not isinstance(texto, str):
            return ""
        padrao = r"(SUBS[ÍI]DIOS\s+(?:AO|PARA\s+O)\s+EXAME\s+T[ÉE]CNICO\b|RAZ[OÕ]ES\b|R\s*A\s*Z\s*[ÕO]\s*E\s*S|Excelent[ií]ssim[oa]\b|Ilustr[ií]ssim[oa]\s|Ilm[oa]\s+Senhor\s+|Ilm[oa]\s+Senhora\b)"
        match = re.search(padrao, texto, flags=re.IGNORECASE | re.DOTALL)
        if match:
            return texto[match.start():]
        else:
            return

    def iniciar_apos_recurso_260(self, texto):
        if not isinstance(texto, str):
            return ""
        padrao = r"(CONTRARRAZ[ÕO]ES\s+(?:AO|SOBRE\s+O)\s+RECURSO\b|CONTRARRAZ[ÕO]ES\s+(?:AO|SOBRE\s+O)\s+EXAME\b|APRESENTA[ÇC][ÃA]O\s+DE\s+CONTRARRAZ[ÕO]ES\b|ESCLARECIMENTOS?\b|E\s*S\s*C\s*L\s*A\s*R\s*E\s*C\s*I\s*M\s*E\s*N\s*T\s*O\s*S?)|RAZ[OÕ]ES\b|R\s*A\s*Z\s*[ÕO]\s*E\s*S"
        match = re.search(padrao, texto, flags=re.IGNORECASE | re.DOTALL)
        if match:
            return texto[match.start():]
        else:
            return

    def iniciar_apos_recurso_281(self, texto):
        if not isinstance(texto, str):
            return ""
        padrao = r"(Cumprimento\s+de\s+Exig[êe]ncia\b|Justificativa\s+de\s+Patente\b|MANIFESTA[CÇ][AÃ]O\s+(?:SOBRE\s+O|SOBRE|A|AO)\s+PARECER\b|MANIFESTA[CÇ][AÃ]O\s+(?:SOBRE|A|À)\s+CI[ÊE]NCIA\b|RAZ[OÕ]ES\b|R\s*A\s*Z\s*[ÕO]\s*E\s*S|ESCLARECIMENTOS?\b|E\s*S\s*C\s*L\s*A\s*R\s*E\s*C\s*I\s*M\s*E\s*N\s*T\s*O\s*S?|Excelent[ií]ssim[oa]\b|Ilustr[ií]ssim[oa]\s|Ilm[oa]\s+Senhor\s+|Ilm[oa]\s+Senhora\b)"
        match = re.search(padrao, texto, flags=re.IGNORECASE | re.DOTALL)
        if match:
            return texto[match.start():]
        else:
            return

    def iniciar_apos_recurso_214(self, texto):
        if not isinstance(texto, str):
            return ""
        padrao = r"(RECURSO\s+do\s+despacho\s+que\s+indeferiu\s+o\s+Pedido\s+de\s+Paten\s*te|Excelent[ií]ssimo\b|Ilustr[ií]ssimo\s+Senhor\b|Ilmo\s+Senhor\s+Presidente\b|recurso\s+ao\s+presidente\s+|apresentar\s+recurso\s+desta\s+decis[aã]o|Em\s+resposta\s+a\s*(?:o|ao)\s+Indeferi\s*-?\s*mento)"
        match = re.search(padrao, texto, flags=re.IGNORECASE | re.DOTALL)
        if match:
            return texto[match.start():]
        else:
            padrao = r"^\s*RECURSO\s+CONTRA\s+INDEFERIMENTO\s*$"
            match = re.search(padrao, texto, flags=re.IGNORECASE | re.MULTILINE)
            if match:
                return texto[match.start():]
            else:
                padrao = r"^\s*RECURSO\s+AO\s+INDEFERIMENTO\s*$"
                match = re.search(padrao, texto, flags=re.IGNORECASE | re.MULTILINE)
                if match:
                    return texto[match.start():]
                else:
                    padrao = r"^\s*INTERPOSICAO\s+DE\s+RECURSO\s+AO\s+INDEFERIMENTO\s*$"
                    match = re.search(padrao, texto, flags=re.IGNORECASE | re.MULTILINE)
                    if match:
                        return texto[match.start():]
                    else:
                        padrao = r"^\s*RECURSO\s+CONTRA\s+INDEFERIMENTO,\s*$"
                        match = re.search(padrao, texto, flags=re.IGNORECASE | re.MULTILINE)
                        if match:
                            return texto[match.start():]
                        else:
                            padrao = r"(RECURSO\s+ADMINISTRATIVO\s+|recurso\s+contra\s+o\s+indeferimento|recurso\s+ao\s+indeferimento|recurso\s+contra\s+decisao\s+de\s+indeferimento)"
                            match = re.search(padrao, texto, flags=re.IGNORECASE | re.DOTALL)
                            if match:
                                return texto[match.start():]
                            else:
                                padrao = r"(ilustrissimos\s+examinadores)"
                                match = re.search(padrao, texto, flags=re.IGNORECASE | re.DOTALL)
                                if match:
                                    return texto[match.start():]
                                else:
                                    padrao = r"(recurso\s+que\s+bastante\s+faz)"
                                    match = re.search(padrao, texto, flags=re.IGNORECASE | re.DOTALL)
                                    if match:
                                        return texto[match.start():]
                                    return ""

import re
import unicodedata

def normalizar(texto):
    texto = texto.replace('\xa0', ' ')  # remove non-breaking space
    texto = unicodedata.normalize('NFKD', texto)
    texto = texto.encode('ASCII', 'ignore').decode('ASCII')  # remove acentos
    #texto = re.sub(r'\s+', ' ', texto)  # colapsa múltiplos espaços/quebras
    return texto
        
documento = """
RECURSO do despacho que indeferiu o Pedido de Patente
Requerente: JOÃO SILVA OLIVEIRA CPF 123.456.789-00
Técnico: RICARDO FREDERICO NICOL
Processo anterior: 123456789
Email: joao@email.com
Assinado digitalmente por servidor INPI.
"""

anonymizer = DataAnonymizer()
documento = normalizar(documento)
#documento = anonymizer.iniciar_apos_recurso(documento)
texto_anon = anonymizer.anonymize_texto(documento)
print("=== DOCUMENTO ANONIMIZADO ===")
print(texto_anon)

=== DOCUMENTO ANONIMIZADO ===

RECURSO do despacho que indeferiu o Pedido de Patente
Requerente: [PESSOA_NATURAL_1][CPF_1]
Tecnico: RICARDO FREDERICO NICOL
Processo anterior: [PROCESSO_1]
Email: [EMAIL_1]


In [ ]:
import PyPDF2, os

arquivo = 'PI1013364_29409161939772965_207.pdf'

nome_sem_extensao = arquivo.replace('.pdf', '')
partes = nome_sem_extensao.split('_')
numero = partes[0]
numnossonumero = partes[1]
tipo = partes[2]

file_path = f"pareceres/peticoes/{numero}_{numnossonumero}_{tipo}.pdf"
print(file_path)
all_text = ''
if os.path.exists(file_path):
    with open(file_path, "rb") as file:
        reader = PyPDF2.PdfReader(file)
        for page in reader.pages:
            all_text += page.extract_text() or ""
else:
    print(f"Arquivo não encontrado: {file_path}")
    
def anonimizacao(texto, tipo):
    texto_anon = ''
    anonymizer = DataAnonymizer()
    documento_normalizado = normalizar(texto) # elimina quebras de linha e caracteres especiais
    documento_inicial = documento_normalizado
    if tipo=='214':
        documento_inicial = anonymizer.iniciar_apos_recurso_214(documento_normalizado) # busca cabeçalho de início
    if tipo=='207':
        documento_inicial = anonymizer.iniciar_apos_recurso_207(documento_normalizado) # busca cabeçalho de início
    if tipo=='210':
        documento_inicial = anonymizer.iniciar_apos_recurso_210(documento_normalizado) # busca cabeçalho de início
    if tipo=='260':
        documento_inicial = anonymizer.iniciar_apos_recurso_260(documento_normalizado) # busca cabeçalho de início
    if tipo=='281':
        documento_inicial = anonymizer.iniciar_apos_recurso_281(documento_normalizado) # busca cabeçalho de início
    texto_anon = anonymizer.anonymize_texto(documento_inicial) # anonimiza referencias
    #print("=== DOCUMENTO ANONIMIZADO ===")
    #print(texto_anon)
    return texto_anon

#print(all_text)
texto_anon = anonimizacao(all_text,tipo)
print(texto_anon)
if texto_anon == '':
    print("Texto inicial não identificado")

In [ ]:
# 1. Faça o download de todas as petições
# 2. No notepad procure as ocorrências de cpf, cnpj e elimine tais trechos manualmente, remova procurações e recibo do sacado
# 3. Verificar o OCR dos arquivos TXT vazios, possivelmente porque eram PDF imagem
# 4. 

In [ ]:
# donwload das peticoes
# varios registros estao com cd_imagem=1
# numnossonumero=29409161953366073 cd_imagem=1
# para corrigir detectar no DBVISUALIZER os casos de 214
# select * from CEPIT_SISCAP.SISCAP_DESPACHOS_PAG where tipo_peticao='214' and cd_imagem>0
# salva em CSV e carrega no arquivo local do XAMPP na tabela despachos_pag
# delete FROM `despachos_pag`  where cd_imagem=0
# rode o algoritmo de correção de cd_imagem
# faça o import de updates.sql
# confira select * from despachos_pag where cd_imagem=0 and tipo_peticao='214';

import os
import json
import requests

#query = '"' + "mysql_query" + '"' ":" + '"' + f" * FROM carga" + '"'
query = '"' + "mysql_query" + '"' ":" + '"' + f" * FROM carga" + '"'
url = f"https://cientistaspatentes.com.br/apiphp/patents/query/?q={query}"
print(url)

json_data = conectar_siscap(url,return_json=True)
json_data = re.sub(r'[\x00-\x08\x0B\x0C\x0E-\x1F]', '', json_data)
json_data = json_data.replace('\r', '')
data = json.loads(json_data)

#json_str = json.dumps(json_data)
#json_str = re.sub(r'[\x00-\x08\x0B\x0C\x0E-\x1F]', '', json_str)
#json_str = json_str.replace('\r', '')
#data = json.loads(json_str)

#print(json_data)

tipo = '272'
#tipo = '207'
#tipo = '210'
#tipo = '260'
#tipo = '281'
#tipo = '207'
#tipo = '214'
#tipo = '200'

#data = {
#    "patents": [
#        {"numero": "102012030377"}
#    ]
#}

for patent in data.get("patents", []):
    numero = patent.get("numero")
    #divisao = patent.get("divisao")
    if not numero or numero == "NUMERO":
        continue

    # https://cientistaspatentes.com.br/apiphp/patents/query/?q="mysql_query":" * FROM despachos_pag WHERE numero='102013033208' and tipo_peticao='272'" 
    json_data = None
    query_pedido = '"' + "mysql_query" + '"' ":" + '"' + f" * FROM despachos_pag where numero='{numero}' and tipo_peticao='{tipo}'"+'"' # 112015029938
    url = f"https://cientistaspatentes.com.br/apiphp/patents/query/?q={query_pedido}"
    print(url)

    numnossonumero = None
    cd_imagem = None
    try:
        json_data = conectar_siscap(url,return_json=True)
    except:
        pass

    if not json_data:
        continue

    #print(json_data)
  
    if json_data:
        try:
            json_data = re.sub(r'[\x00-\x08\x0B\x0C\x0E-\x1F]', '', json_data)
            json_data = json_data.replace('\r', '')
            data_pedido = json.loads(json_data)
            patents = data_pedido.get("patents", [])
            if not patents:
                continue
                
            for primeiro in patents:
                #numnossonumero = data_pedido["patents"][0]["numnossonumero"]
                #cd_imagem = int(data_pedido["patents"][0]["cd_imagem"])
                #data_peticao = data_pedido["patents"][0]["data_peticao"]
                #primeiro = patents[0]
                #if not isinstance(primeiro, dict):
                #    continue
                    
                numnossonumero = primeiro.get("numnossonumero")
                cd_imagem = int(primeiro.get("cd_imagem", 0))
                data_peticao = primeiro.get("data_peticao")
                nova_data = converter_data(data_peticao) if data_peticao else ""
                print (f"numnossonumero={numnossonumero} cd_imagem={cd_imagem} data_peticao={data_peticao}")

                if not cd_imagem or cd_imagem==1 or cd_imagem==0:
                    continue

                url = f"http://br00-aux.inpi.gov.br/webservice/retornaImagem.php?codigo={cd_imagem}"
                arquivo_saida = f"peticoes/{numero}_{numnossonumero}_{tipo}.pdf"

                try:
                    response = requests.get(url, stream=True, verify=False, timeout=30)
                
                    if response.status_code == 200:
                        if not os.path.exists(arquivo_saida):
                            with open(arquivo_saida, "wb") as f:
                                for chunk in response.iter_content(chunk_size=8192):
                                    if chunk:
                                        f.write(chunk)
                            print(f"Download concluído: {arquivo_saida}")
                            
                            all_text = ''
                            if os.path.exists(arquivo_saida):
                                with open(arquivo_saida, "rb") as file:
                                    reader = PyPDF2.PdfReader(file)
                                    for page in reader.pages:
                                        all_text += page.extract_text() or ""
                            else:
                                print(f"Arquivo não encontrado: {file_path}")
                        
                            texto_relatorio = anonimizacao(all_text,tipo)
                            
                            if not texto_relatorio or len(texto_relatorio.strip()) < 50:
                                texto_relatorio = anonimizacao(all_text,'completo') # 112012032204_29409161957989504_214 estava vazio !
                                
                            caminho_do_arquivo = f"peticoes/{numero}_{numnossonumero}_{tipo}.txt"
                            # os.makedirs(os.path.dirname(caminho_do_arquivo), exist_ok=True)

                            resultado = [v for k, v in TIPO_CHOICES if k == tipo][0]
                            output = numero + '\n' + "Petição " + resultado + '\n'
                            output = output + "Data da Petição: " + nova_data + '\n\n'
                            texto_relatorio = output + texto_relatorio
            
                            if not os.path.exists(caminho_do_arquivo):
                                #os.makedirs(os.path.dirname(caminho_do_arquivo), exist_ok=True)
                                print(f"Arquivo novo criado: {caminho_do_arquivo}")
                                with open(caminho_do_arquivo, "w", encoding="utf-8") as arquivo:
                                    arquivo.write(texto_relatorio)
                            else:
                                print(f"Arquivo já existe, não sobrescrito: {caminho_do_arquivo}")   
                        else:
                            print(f"O arquivo {arquivo_saida} já existe. Nada foi gravado.")
                    else:
                        print(f"Falha no download. HTTP {response.status_code}")
                
                except requests.exceptions.RequestException as e:
                    print(f"Erro na requisição: {e}")
   
                    
        except json.JSONDecodeError:
            pass  # JSON inválido → segue o fluxo sem abortar

In [ ]:
tem arquivo que tem cd_imagem zero mas a petição existe

https://cientistaspatentes.com.br/apiphp/patents/query/?q="mysql_query":" * FROM despachos_pag where numero='112016006171' and tipo_peticao='207'"
{"patents": [{"numero":"112016006171","peticao":"WBRJ 870200051285","numnossonumero":"29409161917612280","data_peticao":"2020-04-24",
              "tipo_peticao":"207","flag_pedexame":"1","flag_imagem":"0","cd_imagem":"0","update_imagem":"0000-00-00 00:00:00","conciliado":"1"},
             {"numero":"112016006171","peticao":"WBRJ 870190119058","numnossonumero":"29409161912544902","data_peticao":"2019-11-18",
              "tipo_peticao":"207","flag_pedexame":"1","flag_imagem":"0","cd_imagem":"0","update_imagem":"0000-00-00 00:00:00","conciliado":"1"}]}

numnossonumero=29409161917612280 cd_imagem=0 data_peticao=2020-04-24
numnossonumero=29409161912544902 cd_imagem=0 data_peticao=2019-11-18

no CEPIT temos cd_imagem igualmente NULL
select * from CEPIT_SISCAP.SISCAP_DESPACHOS_PAG where numero='112016006171' and tipo_peticao='207' 


In [ ]:
# rotina que faz OCR de PDF imagem
# por.traineddata https://github.com/tesseract-ocr/tessdata/tree/main
# salve este arquivo em D:/Users/abrantes/AppData/Local/Programs/Tesseract-OCR/tessdata
# consulte Udemy pdfimagem.ipynb
# pip install pdfplumber

import pytesseract
from pdf2image import convert_from_path
from PIL import Image
import pdfplumber
import subprocess


def ocr_pdf_imagem(pdf_path):
    # Definir o caminho do executável do Tesseract (necessário apenas no Windows)
    pytesseract.pytesseract.tesseract_cmd = "C:/Program Files/Tesseract-OCR/tesseract.exe"
    pytesseract.pytesseract.tesseract_cmd = "D:/Users/abrantes/AppData/Local/Programs/Tesseract-OCR/tesseract.exe"
    
    # Caminho para o arquivo PDF
    # instalação do poppler https://github.com/oschwartz10612/poppler-windows/releases
    poppler_path = r'D:\Users\abrantes\poppler-24.08.0\Library\bin'  
    
    # Converter PDF em uma lista de imagens (cada página do PDF será uma imagem)
    pages = convert_from_path(pdf_path, 300, poppler_path=poppler_path)  # 300 DPI para qualidade de imagem
    
    # Percorrer cada página e extrair o texto
    text = ""
    for page in pages:
        # Converter a imagem para texto usando o pytesseract (OCR)
        text += pytesseract.image_to_string(page, lang='por')  # 'lang' define o idioma, 'por' para português
    
    # Exibir o texto extraído
    return text

def eh_pdf_imagem(caminho_pdf):
    with pdfplumber.open(caminho_pdf) as pdf:
        for pagina in pdf.pages:
            texto = pagina.extract_text()
            if texto and texto.strip():
                return False  # tem texto → não é só imagem
    return True  # nenhuma página tinha texto → é PDF imagem
    
pdf_path = 'pareceres/peticoes/PI0800882_29409161909986908_214.pdf'
pdf_path = 'peticoes/PI1013364_0000921108228711_200.pdf'
pdf_path = 'peticoes/202013018647_0000221304137087_200_new.pdf'
pdf_path = 'peticoes/102012030377_0000221208213649_200.pdf'
pdf_path = 'peticoes/MU9002515_29409161711782180_207.pdf'
pdf_path = 'peticoes/102014022484_0000221406041577_200_new.pdf'
pdf_path = 'peticoes/MU9002515_0000221010743893_200.pdf'
pdf_path = 'peticoes/102014015369_0000221404718383_200_new.pdf'
pdf_path = 'peticoes/102013029155_0000221306802568_200_new.pdf'
pdf_path = 'peticoes/MU9002515_29409161901216836_214_new.pdf'
pdf_path = 'peticoes/202013033354_29409161913762261_214_new.pdf'
pdf_path = 'peticoes/PI0800882_29409161909986908_214_new.pdf'
pdf_path = 'peticoes/202014006525_29409161923538828_214_new.pdf'
pdf_path = 'peticoes/202014000433_29409161921972245_214_new.pdf'
pdf_path = 'peticoes/202013017528_29409161909571155_281.pdf'
pdf_path = 'peticoes/202013017528_29409161913950254_281.pdf'
pdf_path = 'peticoes/PI1015975_29409161937794589_214.pdf'

#teste = eh_pdf_imagem (pdf_path)
texto = ocr_pdf_imagem(pdf_path)
print(pytesseract.get_tesseract_version())
print(texto)

In [ ]:
# pule esta instrução
txt_path = 'peticoes/202013018647_0000221304137087_200_new.txt'
tipo = '200'
texto_relatorio = anonimizacao(texto,tipo)
output = numero + '\n' + "Petição " + str(TIPO_CHOICES.get(tipo)) + '\n'
output = output + "Data da Petição: " + nova_data + '\n\n'
texto_relatorio = output + texto_relatorio
print(f"Arquivo novo criado: {txt_path}")
with open(txt_path, "w", encoding="utf-8") as arquivo:
    arquivo.write(texto_relatorio)


In [ ]:
import pypdfium2 as pdfium
import pytesseract

pdf_path = '29409161929498518.pdf' # caracteres estranhos
pdf_path = 'pareceres/peticoes/PI0800882_29409161909986908_214.pdf'
#pdf_path = 'peticoes/PI1013364_0000921108228711_200.pdf'
#pdf_path = 'peticoes/202013018647_0000221304137087_200.pdf'
#pdf_path = 'peticoes/102012030377_0000221208213649_200.pdf'
#pdf_path = 'peticoes/102014022484_0000221406041577_200.pdf'
#pdf_path = 'peticoes/MU9002515_0000221010743893_200.pdf'
#pdf_path = 'peticoes/102014015369_0000221404718383_200.pdf'
#pdf_path = 'peticoes/102013029155_0000221306802568_200.pdf'
#pdf_path = 'peticoes/MU9002515_29409161901216836_214.pdf'
#pdf_path = 'peticoes/202013033354_29409161913762261_214.pdf'
#pdf_path = 'peticoes/102015025507_0000221506838884_200.pdf'
#pdf_path = 'peticoes/PI0800882_29409161909986908_214.pdf'
#pdf_path = 'peticoes/202014006525_29409161923538828_214.pdf'
#pdf_path = 'peticoes/202014000433_29409161921972245_214_new.pdf'
pdf_path = 'peticoes/MU9002515_29409161711782180_207.pdf'
pdf_path = 'peticoes/PI1012905_0000921112640532_200.pdf'
pdf_path = 'peticoes/PI1012651_0000921108899101_200.pdf'
pdf_path = 'peticoes/PI1011888_0000921113196806_200.pdf'
pdf_path = 'peticoes/PI0912882_0000921009358374_200.pdf'
pdf_path = 'peticoes/PI0701396_0000220700730220_200.pdf'
pdf_path = 'peticoes/MU9100724_0000221102157958_200.pdf'
pdf_path = 'peticoes/202012022845_0000221206131491_200.pdf'
pdf_path = 'peticoes/202012008411_0000921202241360_200.pdf'
pdf_path = 'peticoes/MU9001502_3158861707370052_214.pdf'
pdf_path = 'peticoes/102012015992_0000221112317176_200.pdf'
pdf_path = 'peticoes/202012030369_29409161914052993_214.pdf'
pdf_path = 'peticoes/202013013463_29409161813021677_214.pdf'
pdf_path = 'peticoes/122019014139_29409161925875490_281.pdf'
pdf_path = 'peticoes/MU9002515_29409161808161334_281.pdf'
pdf_path = 'peticoes/112014027287_29409161953640850_214.pdf'
pdf_path = 'peticoes/112012026570_29409161711240426_207.pdf'

nome = pdf_path.split('/')[-1]   # pega só o nome do arquivo
tipo = nome.split('_')[2]

pdf = pdfium.PdfDocument(pdf_path)
for i, page in enumerate(pdf):
    
    bitmap = page.render(scale=300/72)
    image = bitmap.to_pil()

    texto = pytesseract.image_to_string(image)

    print(f"\n===== Página {i+1} =====\n")
    print(texto)

In [ ]:
import os

txt_path = os.path.splitext(pdf_path)[0] + '.txt'
texto_relatorio = anonimizacao(texto,tipo)
output = numero + '\n' + "Petição " + str(TIPO_CHOICES.get(tipo)) + '\n'
output = output + "Data da Petição: " + nova_data + '\n\n'
texto_relatorio = output + texto_relatorio
print(f"Arquivo novo criado: {txt_path}")
with open(txt_path, "w", encoding="utf-8") as arquivo:
    arquivo.write(texto_relatorio)

In [ ]:
from pypdf import PdfReader

pdf_path = '29409161929498518.pdf' 
reader = PdfReader(pdf_path)

for page in reader.pages:
    print(page.extract_text())

In [ ]:
from pypdf import PdfReader
import pypdfium2 as pdfium
import pytesseract


def extrair_com_pypdf(pdf_path):

    reader = PdfReader(pdf_path)
    texto = ""

    for page in reader.pages:
        t = page.extract_text()
        if t:
            texto += t

    return texto.strip()


def extrair_com_ocr(pdf_path):

    pdf = pdfium.PdfDocument(pdf_path)
    texto = ""

    for page in pdf:

        bitmap = page.render(scale=400/72)
        image = bitmap.to_pil()

        t = pytesseract.image_to_string(image, lang="por")
        texto += t

    return texto


def extrair_texto_pdf(pdf_path):

    print("Tentando extrair texto com pypdf...")

    texto = extrair_com_pypdf(pdf_path)

    if len(texto) > 50:
        print("Texto encontrado com pypdf.")
        return texto

    print("pypdf falhou. Usando OCR...")

    texto = extrair_com_ocr(pdf_path)

    return texto


# execução

arquivo = 'peticoes/PI1106938_0000221109354295_200.pdf'
texto = extrair_texto_pdf(arquivo)
print("\n=== TEXTO EXTRAÍDO ===\n")
print(texto)

In [ ]:
# verificar o OCR dos arquivos TXT vazios, possivelmente porque eram PDF imagem
import os
import subprocess

print(pytesseract.get_tesseract_version())

arquivo = 'peticoes/102015027652_0000221506193662_200.txt'
tamanho = os.path.getsize(arquivo)
print("Tamanho em bytes:", tamanho)

diretorio = "peticoes"
for raiz, dirs, arquivos in os.walk(diretorio):
    for arquivo in arquivos:
        if arquivo.lower().endswith(".txt"):
            caminho_txt = os.path.join(raiz, arquivo)
            tamanho = os.path.getsize(caminho_txt)
            caminho_pdf = caminho_txt.replace('.txt','.pdf')
    
            if tamanho < 512:  # 1 KB = 1024 bytes
                m = re.search(r'_(\d+)\.txt$', caminho_txt)
                if m:
                    tipo = m.group(1)
                    if tipo != '260':
                        print(f"{caminho_txt} {caminho_pdf} - {tamanho} bytes")
                        texto = extrair_com_ocr(caminho_pdf)       
                        texto_relatorio = anonimizacao(texto,tipo)
                        # resultado = [v for k, v in TIPO_CHOICES if k == tipo][0]
                        # output = numero + '\n' + "Petição " + resultado + '\n'
                        # output = output + "Data da Petição: " + nova_data + '\n\n'
                        # texto_relatorio = output + texto_relatorio
                        print(f"Arquivo novo criado {tipo}: {caminho_txt}")
                        with open(caminho_txt, "a", encoding="utf-8") as arquivo:
                            arquivo.write(texto_relatorio)

In [21]:
# nao execute esta instrução

total = 0
diretorio = "peticoes"
for raiz, dirs, arquivos in os.walk(diretorio):
    for arquivo in arquivos:
        if arquivo.lower().endswith(".txt"):
            caminho_txt = os.path.join(raiz, arquivo)
            m = re.search(r'_(\d+)\.txt$', caminho_txt)
            if m:
                tipo = m.group(1)
                if tipo == '260':
                    print(caminho_txt)
                    total = total + 1
                    os.remove(caminho_txt)
print(total)

0


In [12]:
import re

import os

def remover_recibo_final(texto, limite_linhas=150):
    """
    Remove o bloco 'RECIBO DO SACADO' apenas se ele estiver no final do arquivo.
    Retorna (texto_modificado, houve_corte)
    """

    linhas = texto.splitlines()

    for i, linha in enumerate(linhas):
        if "RECIBO DO SACADO" in linha.upper():

            linhas_restantes = len(linhas) - i

            if linhas_restantes <= limite_linhas:
                return "\n".join(linhas[:i]), True

    return texto, False

def remover_instrucoes_final(texto, limite_linhas=150):
    """
    Remove o bloco 'A data de vencimento nao prevalece sobre o prazo legal' apenas se ele estiver no final do arquivo.
    Retorna (texto_modificado, houve_corte)
    """

    linhas = texto.splitlines()

    for i, linha in enumerate(linhas):
        if "PREVALECE SOBRE O PRAZO LEGAL. O PAGAMENTO DEVE SER EFETUADO" in linha.upper():

            linhas_restantes = len(linhas) - i

            if linhas_restantes <= limite_linhas:
                return "\n".join(linhas[:i]), True

    return texto, False

def remover_procuracao_final(texto, limite_linhas=150):
    """
    Remove o bloco 'PROCURACAO' apenas se ele estiver no final do arquivo.
    Retorna (texto_modificado, houve_corte)
    """

    linhas = texto.splitlines()

    for i, linha in enumerate(linhas):
        if "PROCURACAO" in linha.upper() or "P R O C U R A C A O" in linha.upper() or "P R O C U R A Ç Ã O" in linha.upper() or "PROCURAÇÃO" in linha.upper():

            linhas_restantes = len(linhas) - i

            if linhas_restantes <= limite_linhas:
                return "\n".join(linhas[:i]), True

    return texto, False

def remover_linhas_vazias_multiplas(texto):
    # Substitui 2 ou mais quebras de linha por apenas duas (\n\n = 1 linha em branco)
    return re.sub(r'\n\s*\n+', '\n\n', texto)
    
def processar_txt(entrada, saida):

    with open(entrada, "r", encoding="utf-8") as f:
        texto = f.read()

    texto_limpo, houve_corte = remover_recibo_final(texto)

    if not houve_corte:
        print("Pulando arquivo (recibo não está no final):", entrada)
        texto_limpo, houve_corte = remover_instrucoes_final(texto)
        if not houve_corte:
            print("Pulando arquivo (instruções não está no final):", entrada)
            texto_limpo, houve_corte = remover_procuracao_final(texto)
            if not houve_corte:
                print("Pulando arquivo (procuração não está no final):", entrada)
                texto_limpo = remover_linhas_vazias_multiplas(texto_limpo)
                with open(saida, "w", encoding="utf-8") as f:
                    f.write(texto_limpo)
                return
                
    texto_limpo = remover_linhas_vazias_multiplas(texto_limpo)
    with open(saida, "w", encoding="utf-8") as f:
        f.write(texto_limpo)

    print("Arquivo criado:", saida)


arquivo_entrada = "peticoes/102013033387_29409161940566800_214.txt"  # RECIBO DO SACADO AO FINAL
arquivo_entrada = "peticoes/102012004873_29409161938119486_281.txt"  # INSTRUCOES AO FINAL
arquivo_entrada = "peticoes/102012021084_29409161946481776_281.txt"  # PROCURACAO AO FINAL
arquivo_entrada = "peticoes/PI1103116_29409161929427670_214.txt" # com vários pula linha

diretorio = os.path.dirname(arquivo_entrada)
nome_arquivo = os.path.basename(arquivo_entrada)
#arquivo_saida = os.path.join(diretorio, "new_" + nome_arquivo)
arquivo_saida = arquivo_entrada
processar_txt(arquivo_entrada, arquivo_saida)

Pulando arquivo (recibo não está no final): peticoes/PI1103116_29409161929427670_214.txt
Pulando arquivo (instruções não está no final): peticoes/PI1103116_29409161929427670_214.txt
Pulando arquivo (procuração não está no final): peticoes/PI1103116_29409161929427670_214.txt


In [ ]:
diretorio = "peticoes"
for nome_arquivo in os.listdir(diretorio):
    if nome_arquivo.lower().endswith(".txt"):
        arquivo_entrada = os.path.join(diretorio, nome_arquivo)
        arquivo_saida = arquivo_entrada
        processar_txt(arquivo_entrada, arquivo_saida)

In [1]:
from langchain_community.llms import Ollama
llm = Ollama(
    model="phi3",
    temperature=0.2,
)
response = llm.invoke("Explique a diferença entre Durkheim e Weber")

D:\Users\abrantes\AppData\Local\Temp\ipykernel_3184\1675997290.py:2: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaLLM``.
  llm = Ollama(


In [3]:
print(response)

É importante notar que Émile Durkheim (1854-1917) e Max Weber (1864-1920), embora ambos sejam figuras fundamentais na sociologia, têm abordagens distintas para entender a sociedade.

Durkheim é conhecido por sua ênfase no papel da "coesão social" e nas normas sociais que unem as pessoas em uma comunidade ou sociedade maior. Durkheim argumenta que essas normas são fundamentais para a manutenção de um estado estável, pois fornecem direções claras sobre como os indivíduos devem agir e reagir uns aos outros dentro da mesma comunidade ou sociedade. Ele também enfatiza o papel das instituições sociais na formação do comportamento individual, sugerindo que essas instituições moldam as crenças e ações dos indivíduos de maneiras profundamente enraizadas em sua cultura social.

Por outro lado, Max Weber (1864-1920) se concentra na "lógica da vida cotidiana", examinando como as pessoas vivem suas vidas e interagem uns com os outros dentro de uma sociedade específica. Ele enfatiza o papel das cren

In [ ]:
arquivo_entrada = "peticoes/102015020759_29409161922565716_281.txt"
texto = ''
with open(arquivo_entrada, "r", encoding="utf-8") as f:
    texto = f.read()
print(texto)

In [9]:
prompt = f""" Esta é uma petição usada em um processo administrativo. Eu desejo limpar esta petição retirando os textos que
se referem a procurações, recibos, documentos de cessão, documentos de prioridade, depósitos PCT e formulários de entrada 
que não dizem respeito ao conteúdo em si da petição. O texto final recuperado deve conter apenas os argumentos do requerente, 
textos de relatório descrivo, reivindicações e resumo do pedido de patente. Se o requerente reapresenta relatório descritivo
e revindicações e resumo quero que isso seja mantido no texto final. Qualquer referência a dadaos pessoais como CPF, CNPJ, número de OAB e 
CREA bem como endereços deve ser eliminado do texto. Obtenha como saída apenas o texto recuperado em sua forma original, sem resumir nem traduzir nada. 
O texto da petição original é o seguinte: {texto}"""
response = llm.invoke(prompt)
print(response)

The document provided appears to be a detailed description of various agricultural vehicle configurations that integrate traction control systems with the axle and wheel assembly, specifically focusing on an arrangement wherein both front-wheel drive (FWD) or rear-wheel drive vehicles are equipped with such technology. The descriptions cover multiple aspects including structural components like mounting brackets for differentials, hydra0accelerator gearboxes, and the inclusion of a motorized transmission system within certain models; they also touch upon specific design elements that enhance functionality in agricultural settings—such as traction control systems (TCS), acoustic dampers on axles to reduce noise from differentials, locking hub assemblies for improved stability during field operations, and the integration of a motorized transmission within certain models.

To ensure clarity and maintain consistency in technical language throughout this document while removing any referenc